# Silver — ecommerce_produtos / ecommerce_categorias
Squad 1 · Dupla 3 (Bruna Diniz & Rafael Miranda)

**Regras/KPIs gerados por este arquivo:** as 20 regras de qualidade
oficiais (10 produtos + 10 categorias), como coluna booleana por linha
(`regra_01` a `regra_10`), gravadas em `squad1/silver/` em append.

**ecommerce_produtos:** 1. sku_valido · 2. preco_positivo ·
3. nome_preenchido · 4. categoria_existe · 5. unidade_valida ·
6. is_ativo_valido · 7. preco_dentro_limite · 8. marca_preenchida ·
9. ativo_com_venda_90d (**provisório**, consome a Raw direto, autorizado
pelo Geovany em 01/10) · 10. pertence_subcategoria

**ecommerce_categorias:** 1. id_valido · 2. nome_preenchido · 3. pai_existe ·
4. sem_ciclo_direto · 5. nome_unico_no_nivel · 6. dois_niveis ·
7. subcategoria_tem_produto (**pendente**) · 8. raiz_tem_subcategoria ·
9. nome_sem_caracteres_invalidos (`&` é permitido, confirmado pelo Geovany
em 28/09) · 10. total_raizes_confere (**pendente** — falta valor de
referência do config.py)

In [0]:
%pip install deltalake --quiet
dbutils.library.restartPython()

In [0]:
import os
import io
import pandas as pd
from dotenv import load_dotenv
from pyspark.sql import functions as F
from pyspark.sql import Window
from deltalake import DeltaTable, write_deltalake
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

load_dotenv(".env")

TENANT_ID = os.getenv("ADLS_TENANT_ID")
CLIENT_ID = os.getenv("ADLS_CLIENT_ID")
CLIENT_SECRET = os.getenv("ADLS_CLIENT_SECRET")

STORAGE_ACCOUNT = "internshipdatalake"
SQUAD_CONTAINER = "squad1"
lakehouse_base = f"abfss://{SQUAD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

bronze_produtos_path = f"{lakehouse_base}/bronze/ecommerce_produtos"
bronze_categorias_path = f"{lakehouse_base}/bronze/ecommerce_categorias"
silver_produtos_path = f"{lakehouse_base}/silver/ecommerce_produtos"
silver_categorias_path = f"{lakehouse_base}/silver/ecommerce_categorias"

storage_options = {
    "azure_storage_account_name": STORAGE_ACCOUNT,
    "azure_storage_client_id": CLIENT_ID,
    "azure_storage_client_secret": CLIENT_SECRET,
    "azure_storage_tenant_id": TENANT_ID,
}
adls_options = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": CLIENT_ID,
    "fs.azure.account.oauth2.client.secret": CLIENT_SECRET,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token",
}
credential = ClientSecretCredential(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
datalake_service = DataLakeServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT}.dfs.core.windows.net",
    credential=credential
)
squad1_fs_client = datalake_service.get_file_system_client(SQUAD_CONTAINER)
raw_fs_client = datalake_service.get_file_system_client("raw")

def ler_delta_direto(caminho):
    dt = DeltaTable(caminho, storage_options=storage_options)
    return spark.createDataFrame(dt.to_pandas())

print("Setup concluído.")

`regras_categorias`: calcula as 10 regras de categorias e marca cada linha
com `regra_01` a `regra_10` (as técnicas 07 e 10 ainda ficam `None` —
pendentes).

In [0]:
def regras_categorias(df):
    df_ref = df.select("bronze_source_file", "id_categoria", "id_categoria_pai").dropDuplicates(["bronze_source_file", "id_categoria"])

    w_dup = Window.partitionBy("bronze_source_file", "id_categoria")
    df = df.withColumn("_id_count", F.count("id_categoria").over(w_dup))

    df = (df
        .withColumn("regra_01_id_valido",
            F.col("id_categoria").isNotNull() & (F.col("_id_count") == 1))
        .withColumn("regra_02_nome_preenchido",
            F.col("nome_categoria").isNotNull() & (F.trim(F.col("nome_categoria")) != ""))
        .withColumn("regra_04_sem_ciclo_direto",
            F.col("id_categoria_pai").isNull() | (F.col("id_categoria_pai") != F.col("id_categoria")))
        .withColumn("regra_09_nome_sem_caracteres_invalidos",
            ~F.col("nome_categoria").rlike(r'[<>]|script'))
    ).drop("_id_count")

    w_nivel = Window.partitionBy("bronze_source_file", F.coalesce(F.col("id_categoria_pai"), F.lit(-1)), "nome_categoria")
    df = df.withColumn("_nome_nivel_count", F.count("nome_categoria").over(w_nivel))
    df = df.withColumn("regra_05_nome_unico_no_nivel", F.col("_nome_nivel_count") == 1).drop("_nome_nivel_count")

    ids_por_arquivo = df_ref.groupBy("bronze_source_file").agg(F.collect_set("id_categoria").alias("_ids_validos"))
    df = df.join(ids_por_arquivo, "bronze_source_file", "left")
    df = df.withColumn(
        "regra_03_pai_existe",
        F.col("id_categoria_pai").isNull() | F.array_contains(F.col("_ids_validos"), F.col("id_categoria_pai"))
    ).drop("_ids_validos")

    df_pais = df_ref.select(
        F.col("bronze_source_file").alias("_pf"),
        F.col("id_categoria").alias("_id"),
        F.col("id_categoria_pai").alias("_pai_do_pai")
    )
    df = (df
        .join(df_pais, (df["bronze_source_file"] == F.col("_pf")) & (df["id_categoria_pai"] == F.col("_id")), "left")
        .withColumn("regra_06_dois_niveis",
            F.col("id_categoria_pai").isNull() | F.col("_pai_do_pai").isNull())
        .drop("_pf", "_id", "_pai_do_pai")
    )

    subcats_por_arquivo = (df_ref.filter(F.col("id_categoria_pai").isNotNull())
        .groupBy("bronze_source_file").agg(F.collect_set("id_categoria_pai").alias("_pais_com_filho")))
    df = df.join(subcats_por_arquivo, "bronze_source_file", "left")
    df = df.withColumn(
        "regra_08_raiz_tem_subcategoria",
        F.when(F.col("id_categoria_pai").isNotNull(), F.lit(True))
         .otherwise(F.array_contains(F.col("_pais_com_filho"), F.col("id_categoria")))
    ).drop("_pais_com_filho")

    df = df.withColumn("regra_07_subcategoria_tem_produto", F.lit(None).cast("boolean"))
    df = df.withColumn("regra_10_total_raizes_confere", F.lit(None).cast("boolean"))

    return df

`regras_produtos`: calcula as 10 regras de produtos. Recebe a Silver de
categorias (pra checar FK) e a lista de SKUs vendidos nos últimos 90 dias
(pra `regra_09`).

In [0]:
def regras_produtos(df, df_categorias_silver, skus_vendidos_90d):
    df_cat_ref = df_categorias_silver.select("id_categoria", "id_categoria_pai").dropDuplicates(["id_categoria"])

    w_sku = Window.partitionBy("bronze_source_file", "sku")
    df = df.withColumn("_sku_count", F.count("sku").over(w_sku))

    df = (df
        .withColumn("regra_01_sku_valido",
            F.col("sku").isNotNull() & (F.col("_sku_count") == 1))
        .withColumn("regra_02_preco_positivo",
            F.col("preco_lista").isNotNull() & (F.col("preco_lista") > 0))
        .withColumn("regra_03_nome_preenchido",
            F.col("nome_produto").isNotNull() & (F.trim(F.col("nome_produto")) != ""))
        .withColumn("regra_05_unidade_valida",
            F.col("unidade_medida").isin("un", "kg", "g", "L", "ml"))
        .withColumn("regra_06_is_ativo_valido",
            F.col("is_ativo").isNotNull())
        .withColumn("regra_07_preco_dentro_limite",
            F.col("preco_lista").isNotNull() & (F.col("preco_lista") <= 5000))
        .withColumn("regra_08_marca_preenchida",
            F.col("nome_marca").isNotNull() & (F.trim(F.col("nome_marca")) != ""))
    ).drop("_sku_count")

    df = (df
        .join(df_cat_ref, "id_categoria", "left")
        .withColumn("regra_04_categoria_existe", F.col("id_categoria_pai").isNotNull() | F.col("id_categoria").isNotNull())
        .withColumn("regra_10_pertence_subcategoria", F.col("id_categoria_pai").isNotNull())
    )

    # PROVISÓRIO (autorizado pelo Geovany, 01/10/2026): skus_vendidos_90d vem
    # direto da Raw (ecommerce_pedidos + ecommerce_itens_pedido), já que a
    # Silver de itens_pedido (Dupla 1) ainda não existe.
    # TODO: trocar para consumir da Silver da Dupla 1 quando disponível.
    df = df.withColumn(
        "regra_09_ativo_com_venda_90d",
        F.when(F.col("is_ativo") != True, F.lit(True))
         .otherwise(F.col("sku").isin(skus_vendidos_90d))
    )

    return df

`processar_silver`: lê o Bronze, filtra só o que ainda não está na Silver
(por `bronze_source_file`), aplica as regras e grava em Delta, por tabela.

In [0]:
def processar_silver(caminho_bronze, caminho_silver, aplicar_regras):
    df_bronze = ler_delta_direto(caminho_bronze)

    try:
        dt_silver = DeltaTable(caminho_silver, storage_options=storage_options)
        ja_processados = set(dt_silver.to_pandas(columns=["bronze_source_file"])["bronze_source_file"].unique())
        df_micro_lote = df_bronze.filter(~F.col("bronze_source_file").isin(ja_processados))
    except Exception as e:
        print(f"[AVISO] Não foi possível ler Silver existente em {caminho_silver}: {e}")
        print("[AVISO] Tratando TODO o conteúdo da Bronze como pendente. Confirme se isso é esperado!")
        df_micro_lote = df_bronze

    df_silver = aplicar_regras(df_micro_lote).withColumn(
        "silver_processed_at", F.current_timestamp()
    )

    df_pandas = df_silver.toPandas()
    df_pandas["bronze_ingested_at"] = df_pandas["bronze_ingested_at"].dt.tz_localize("UTC")
    df_pandas["silver_processed_at"] = df_pandas["silver_processed_at"].dt.tz_localize("UTC")

    for coluna in df_pandas.columns:
        if coluna.startswith("regra_"):
            df_pandas[coluna] = df_pandas[coluna].astype("boolean")

    write_deltalake(caminho_silver, df_pandas, mode="append", storage_options=storage_options)

    print(f"OK [{caminho_silver}] {len(df_pandas)} linhas processadas")
    return df_silver

**Cálculo de `skus_vendidos_90d` (provisório):** consome a Raw diretamente
(`ecommerce_pedidos` e `ecommerce_itens_pedido`), porque a Silver de
itens_pedido (Dupla 1) ainda não existe — autorizado pelo Geovany em
01/10/2026. `ecommerce_pedidos` é lido via pandas, não Spark, porque a
coluna `dt_pedido` está em `TIMESTAMP(NANOS)`, formato que o leitor Parquet
nativo do Spark rejeita (`PARQUET_TYPE_ILLEGAL`).

**TODO:** trocar para consumir da Silver da Dupla 1 quando publicarem.

In [0]:
# NOTA TÉCNICA: ecommerce_pedidos é lido via pandas/pyarrow, não via Spark.
# Motivo: a coluna dt_pedido (e outras colunas de data da tabela) está no
# formato TIMESTAMP(NANOS), que o leitor Parquet nativo do Spark não suporta
# -> erro [PARQUET_TYPE_ILLEGAL] Illegal Parquet type: INT64 (TIMESTAMP(NANOS,false))
# Não é uma escolha de estilo: é a única forma de ler esse arquivo específico
# hoje. ecommerce_itens_pedido não tem essa limitação e por isso é lido via
# Spark nativo (ler_itens_pedido_spark abaixo).

raw_base_url = f"abfss://raw@{STORAGE_ACCOUNT}.dfs.core.windows.net"

def ler_parquet_raw_pandas(caminho_arquivo):
    file_client = raw_fs_client.get_file_client(caminho_arquivo)
    conteudo = file_client.download_file().readall()
    return pd.read_parquet(io.BytesIO(conteudo))

def listar_arquivos_raw(nome_arquivo):
    paths = raw_fs_client.get_paths(path="real-time-data", recursive=True)
    return [
        p.name for p in paths
        if not p.is_directory
        and p.name.endswith(f"{nome_arquivo}.parquet")
        and "/processado/" not in p.name
    ]

def ler_todos_parquet_raw_pandas(nome_arquivo):
    arquivos = listar_arquivos_raw(nome_arquivo)
    dfs = [ler_parquet_raw_pandas(a) for a in arquivos]
    return pd.concat(dfs, ignore_index=True)

def ler_itens_pedido_spark():
    arquivos = listar_arquivos_raw("ecommerce_itens_pedido")
    caminhos = [f"{raw_base_url}/{a}" for a in arquivos]
    return spark.read.format("parquet").options(**adls_options).load(caminhos)

# PROVISÓRIO (autorizado pelo Geovany em 01/10/2026): consumindo direto da Raw
# porque a Silver de ecommerce_itens_pedido (Dupla 1) ainda não está pronta.
# TODO: trocar para consumir da Silver da Dupla 1 assim que estiver disponível.
df_pedidos_raw = ler_todos_parquet_raw_pandas("ecommerce_pedidos")

data_limite = pd.Timestamp.now() - pd.Timedelta(days=90)
pedidos_90d = set(
    df_pedidos_raw[df_pedidos_raw["dt_pedido"] >= data_limite]["id_pedido"].unique().tolist()
)

df_itens_pedido_spark = ler_itens_pedido_spark()

skus_vendidos_90d = [
    row["sku"] for row in (
        df_itens_pedido_spark
        .filter(F.col("id_pedido").isin(pedidos_90d))
        .select("sku").distinct()
        .collect()
    )
    if row["sku"] is not None
]

print(f"Pedidos nos últimos 90 dias: {len(pedidos_90d)}")
print(f"SKUs distintos vendidos nos últimos 90 dias: {len(skus_vendidos_90d)}")



In [0]:
processar_silver(bronze_categorias_path, silver_categorias_path, regras_categorias)
df_categorias_silver = ler_delta_direto(silver_categorias_path)
processar_silver(
    bronze_produtos_path,
    silver_produtos_path,
    lambda df: regras_produtos(df, df_categorias_silver, skus_vendidos_90d)
)

print("Bronze produtos:", ler_delta_direto(bronze_produtos_path).count())
print("Bronze categorias:", ler_delta_direto(bronze_categorias_path).count())
print("Silver produtos:", DeltaTable(silver_produtos_path, storage_options=storage_options).to_pandas().shape[0])
print("Silver categorias:", DeltaTable(silver_categorias_path, storage_options=storage_options).to_pandas().shape[0])

In [0]:
from deltalake import DeltaTable

TABELAS = {
    "Bronze produtos":   f"{lakehouse_base}/bronze/ecommerce_produtos",
    "Bronze categorias": f"{lakehouse_base}/bronze/ecommerce_categorias",
    "Silver produtos":   f"{lakehouse_base}/silver/ecommerce_produtos",
    "Silver categorias": f"{lakehouse_base}/silver/ecommerce_categorias",
    "dq_monitoring_logs": f"{lakehouse_base}/dq_monitoring_logs",
    "Gold resumo por regra":  f"{lakehouse_base}/gold/dq_resumo_por_regra",
    "Gold resumo por tabela": f"{lakehouse_base}/gold/dq_resumo_por_tabela",
}

def ver_tabela(nome, caminho, n=10):
    try:
        dt = DeltaTable(caminho, storage_options=storage_options)
    except Exception as e:
        print(f"{nome}: ainda não existe ({type(e).__name__})")
        return
    df = dt.to_pandas()
    print(f"{nome}: {len(df)} linhas | colunas: {list(df.columns)}")
    display(df.head(n))

for nome, caminho in TABELAS.items():
    ver_tabela(nome, caminho)